# Historical dataset sanity

Charts and summary tables for one published `historical-payments` dataset version: scenario mix, skew, cardinality, the schema cutover, duplicate exports and physical layout.

Export the historical-source keys before starting Jupyter:

```bash
export AWS_ACCESS_KEY_ID=<HISTORICAL_RUSTFS_ACCESS_KEY>
export AWS_SECRET_ACCESS_KEY=<HISTORICAL_RUSTFS_SECRET_KEY>
```

Set `DATASET` to read another version or a local directory.

In [ ]:
import json
import os

import duckdb
import matplotlib.pyplot as plt

DATASET = os.environ.get("DATASET", "s3://synthetic-source/historical/v10").rstrip("/")

db = duckdb.connect()
db.execute("SET TimeZone = 'UTC'")
if DATASET.startswith("s3://"):
    if not (os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_SECRET_ACCESS_KEY")):
        raise RuntimeError("Export AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY, then restart the kernel.")
    db.execute(
        "CREATE SECRET (TYPE s3, KEY_ID ?, SECRET ?, ENDPOINT 'minio.home.phuchoang.sbs', URL_STYLE 'path', REGION 'us-east-1')",
        [os.environ["AWS_ACCESS_KEY_ID"], os.environ["AWS_SECRET_ACCESS_KEY"]],
    )
manifest = json.loads(db.sql(f"SELECT content FROM read_text('{DATASET}/manifest.json')").fetchone()[0])
events = f"read_parquet('{DATASET}/payment_events/*/*.parquet', union_by_name = true, hive_partitioning = false)"
labels = f"read_parquet('{DATASET}/fraud_labels/*/*.parquet', hive_partitioning = false)"
print(manifest["dataset_version"], manifest["validation"], "history end", manifest["history_end"])

## Volume and scenario mix

Fraud rate per scenario uses simulation truth per attempt; chargebacks are the delayed labels training will see.

In [ ]:
db.sql(f"""
SELECT scenario, count(*) AS attempts, round(100 * count(*) / sum(count(*)) OVER (), 3) AS pct_attempts,
       count(*) FILTER (is_fraud) AS fraud_attempts
FROM {labels} WHERE source = 'LABEL_SOURCE_SIMULATION_TRUTH' GROUP BY ALL ORDER BY attempts DESC
""").show()
db.sql(f"""
SELECT count(*) AS event_rows, count(DISTINCT event_id) AS unique_events,
       round(1 - count(DISTINCT event_id) / count(*), 4) AS duplicate_rate,
       count(DISTINCT payment_id) AS payments,
       (SELECT count(*) FROM {labels} WHERE source = 'LABEL_SOURCE_CHARGEBACK') AS chargebacks
FROM {events}
""").show()
print("configured duplicate rate", manifest["configured"]["duplicate_rate"])

## Outcomes and decision policy

Approval, risk and processor declines per scenario under `sim-rules-v1`.

In [ ]:
db.sql(f"""
WITH outcome AS (
  SELECT coalesce(attempt_declined.attempt_id, payment_succeeded.attempt_id) AS attempt_id,
         CASE WHEN payment_succeeded IS NOT NULL THEN 'succeeded' ELSE attempt_declined.source END AS result
  FROM (SELECT DISTINCT * FROM {events}) WHERE attempt_declined IS NOT NULL OR payment_succeeded IS NOT NULL
)
SELECT scenario, result, count(*) AS attempts
FROM {labels} l JOIN outcome USING (attempt_id)
WHERE source = 'LABEL_SOURCE_SIMULATION_TRUTH' GROUP BY ALL ORDER BY scenario, attempts DESC
""").show(max_rows=40)

## Amounts, categories and merchant skew

In [ ]:
created = db.sql(f"""
SELECT DISTINCT ON (event_id) payment_id,
       payment_created.snapshot.items[1].category AS category,
       payment_created.snapshot.total.minor_units / 100.0 AS amount,
       payment_created.merchant.merchant_id AS merchant
FROM {events} WHERE payment_created IS NOT NULL
""").df()
assert created.payment_id.is_unique, "one payment_created event per payment"
print(len(created), "payments")
shares = created.groupby("merchant").size().sort_values(ascending=False).cumsum() / len(created)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
created.boxplot(column="amount", by="category", ax=left, showfliers=False)
left.set_title("Checkout amount by category (USD, no outliers)")
left.set_xlabel("")
right.plot(range(1, len(shares) + 1), shares.values)
right.set_xscale("log")
right.set_title("Cumulative payment share by merchant rank")
right.set_xlabel("merchant rank")
fig.suptitle("")
plt.tight_layout()
created.groupby("category").amount.describe().round(2)

## Cardinality

Approximate distinct counts of the high-cardinality identifiers.

In [ ]:
db.sql(f"""
SELECT approx_count_distinct(payment_created.customer.customer_id) AS customers,
       approx_count_distinct(payment_created.snapshot.shipping.address_fingerprint) AS addresses,
       approx_count_distinct(attempt_started.signals.device_fingerprint) AS devices,
       approx_count_distinct(attempt_started.signals.ip_network_fingerprint) AS ip_networks,
       approx_count_distinct(attempt_started.signals.payment_method.fingerprint) AS payment_methods
FROM {events}
""").show()
print("configured pools", manifest["configured"]["cardinality"])
if "cardinality" in manifest["observed"]:
    print(json.dumps(manifest["observed"]["cardinality"], indent=2))
    print("Manifest estimates use exported non-null field rows, including duplicate exports, as denominators.")

## Schema evolution

`user_agent_family` is physically absent before the cutover; a merged read shows nulls only there.

In [ ]:
print("cutover", manifest["configured"]["schema_cutover_date"])
db.sql(f"""
SELECT CAST(occurred_at AS DATE) >= DATE '{manifest["configured"]["schema_cutover_date"]}' AS after_cutover,
       min(schema_version) AS min_schema, max(schema_version) AS max_schema,
       count(*) AS attempts, count(attempt_started.signals.user_agent_family) AS with_user_agent
FROM {events} WHERE attempt_started IS NOT NULL GROUP BY ALL ORDER BY ALL
""").show()

## Label delays

Chargebacks arrive days after the payment; the tail lands after the history end.

In [ ]:
db.sql(f"""
SELECT round(date_diff('hour', occurred_at, label_available_at) / 24.0) AS delay_days, count(*) AS chargebacks,
       count(*) FILTER (label_available_at > TIMESTAMPTZ '{manifest["history_end"]}') AS after_history_end
FROM {labels} WHERE source = 'LABEL_SOURCE_CHARGEBACK' GROUP BY ALL ORDER BY ALL
""").show(max_rows=30)

## Physical layout

Files, row groups and compressed bytes per column, from Parquet metadata. Use this before changing encodings.

In [ ]:
print(json.dumps({name: manifest["storage"][name] for name in ("payment_events", "fraud_labels")}, indent=1))
db.sql(f"""
SELECT path_in_schema AS column_path, any_value(encodings) AS encodings,
       round(sum(total_compressed_size) / 2^20, 2) AS compressed_mib,
       round(sum(total_compressed_size) / sum(total_uncompressed_size), 3) AS ratio
FROM parquet_metadata('{DATASET}/payment_events/*/*.parquet')
GROUP BY ALL ORDER BY compressed_mib DESC LIMIT 15
""").show(max_width=200)
db.sql(f"""
SELECT count(DISTINCT file_name) AS files, round(min(size) / 2^20, 2) AS min_mib, round(median(size) / 2^20, 2) AS median_mib,
       round(max(size) / 2^20, 2) AS max_mib
FROM (SELECT file_name, sum(total_compressed_size) AS size FROM parquet_metadata('{DATASET}/payment_events/*/*.parquet') GROUP BY ALL)
""").show()